In [1]:
!pip install --quiet qiskit
!pip install --quiet mqt.bench

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 20.6 MB/s eta 0:00:00


In [2]:
!wget https://packages.microsoft.com/config/ubuntu/20.04/packages-microsoft-prod.deb -O packages-microsoft-prod.deb
!dpkg -i packages-microsoft-prod.deb
!apt-get update
!apt-get install -y dotnet-sdk-8.0

!pip install --quiet qsharp azure-quantum

--2026-01-26 02:20:34--  https://packages.microsoft.com/config/ubuntu/20.04/packages-microsoft-prod.deb
Resolving packages.microsoft.com (packages.microsoft.com)... 13.107.213.67, 13.107.246.67, 2620:1ec:46::67, ...
Connecting to packages.microsoft.com (packages.microsoft.com)|13.107.213.67|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3690 (3.6K) [application/octet-stream]
Saving to: ‘packages-microsoft-prod.deb’

packages-microsoft- 100%[===================>]   3.60K  --.-KB/s    in 0s      

2026-01-26 02:20:34 (1.72 GB/s) - ‘packages-microsoft-prod.deb’ saved [3690/3690]

Selecting previously unselected package packages-microsoft-prod.
(Reading database ... 129073 files and directories currently installed.)
Preparing to unpack packages-microsoft-prod.deb ...
Unpacking packages-microsoft-prod (1.0-ubuntu20.04.1) ...
Setting up packages-microsoft-prod (1.0-ubuntu20.04.1) ...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 

In [3]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qsharp.estimator import EstimatorParams, ErrorBudgetPartition, LogicalCounts
from qsharp.interop.qiskit import estimate
from mqt.bench import get_benchmark, BenchmarkLevel

QISKIT_STD_GATES = [
    "p", "x", "y", "z", "h", "s", "sdg", "t", "tdg", "sx",
    "rx", "ry", "rz", "cx", "cy", "cz", "cp", "crx", "cry",
    "crz", "ch", "ccx", "cu", "id", "u1", "u2", "u3"
]

def estimate_resources(logical_counts, error_budget_dist):
    """Estimates physical resources for a given error budget distribution."""
    params = EstimatorParams()
    params.error_budget = ErrorBudgetPartition()
    params.error_budget.logical = error_budget_dist['logical']
    params.error_budget.t_states = error_budget_dist['t_states']
    params.error_budget.rotations = error_budget_dist['rotations']

    result = logical_counts.estimate(params=params)

    physical_qubits = result["physicalCounts"]["physicalQubits"]
    runtime = result["physicalCounts"]["runtime"]

    return {
        'physical_qubits': physical_qubits,
        'runtime': runtime,
        'metric': runtime * physical_qubits
    }

def compute_cost(qubits, runtime, weight_qubits=0.5):
    """Compute cost with proper normalization."""
    q = max(qubits, 1)
    r = max(runtime, 1)
    cost = (q ** weight_qubits) * (r ** (1 - weight_qubits))
    return cost

def generate_neighbor_adaptive(current, temperature, initial_temp, iteration, 
                                max_iterations, rng):
    """Generate neighbor with adaptive multi-phase perturbation strategy."""
    progress = iteration / max_iterations
    temp_ratio = temperature / initial_temp
    
    if progress < 0.3:
        base_step = 0.7 * temp_ratio + 0.3
        if rng.random() < 0.2:
            base_step = 0.9
    elif progress < 0.7:
        base_step = 0.5 * temp_ratio + 0.2
    else:
        base_step = 0.3 * temp_ratio + 0.05
    
    neighbor = current.copy()
    
    if progress < 0.5:
        dims_to_perturb = 3
    else:
        dims_to_perturb = rng.integers(1, 4)
    
    indices = rng.choice(3, size=dims_to_perturb, replace=False)
    
    for i in indices:
        perturbation = rng.uniform(-base_step, base_step)
        neighbor[i] += perturbation
    
    neighbor = np.maximum(neighbor, 0.01)
    neighbor = neighbor / np.sum(neighbor)
    
    return neighbor

def adaptive_cooling(temperature, initial_temp, improvement_rate, iteration):
    """Adaptive cooling that slows down when improvements are found."""
    base_rate = 0.98
    
    if improvement_rate > 0.01:
        cooling_factor = 0.99
    elif improvement_rate < 0.001:
        cooling_factor = 0.96
    else:
        cooling_factor = base_rate
    
    if iteration % 100 == 0 and iteration > 0:
        return temperature * cooling_factor * 1.05
    
    return temperature * cooling_factor

def generate_lhs_samples(n_samples, n_dims, rng):
    """Generate diverse initial points using Latin Hypercube Sampling."""
    samples = np.zeros((n_samples, n_dims))
    
    for dim in range(n_dims):
        intervals = np.linspace(0, 1, n_samples + 1)
        
        for i in range(n_samples):
            samples[i, dim] = rng.uniform(intervals[i], intervals[i + 1])
        
        rng.shuffle(samples[:, dim])
    
    normalized_samples = np.zeros_like(samples)
    for i in range(n_samples):
        sample = samples[i]
        sample = np.exp(sample)
        normalized_samples[i] = sample / np.sum(sample)
    
    return normalized_samples

def check_convergence(cost_history, window_size=50, threshold=0.001):
    """Check if optimization has converged based on recent cost improvements."""
    if len(cost_history) < window_size:
        return False
    
    recent_costs = cost_history[-window_size:]
    best_recent = min(recent_costs)
    worst_recent = max(recent_costs)
    
    if worst_recent == 0:
        return True
    
    relative_improvement = (worst_recent - best_recent) / worst_recent
    
    return relative_improvement < threshold

def simulated_annealing_optimization(logical_counts, total_budget,
                                     max_iterations=2000,
                                     initial_temp=10.0,
                                     n_restarts=10,
                                     weight_qubits=0.5,
                                     use_adaptive=True):
    """Enhanced Simulated Annealing with adaptive strategies."""
    print(f"\n{'='*70}")
    print("Enhanced Simulated Annealing with Adaptive Strategies")
    print(f"{'='*70}")
    print(f"Initial temperature: {initial_temp}")
    print(f"Max iterations: {max_iterations}")
    print(f"Restarts: {n_restarts}")
    print(f"Adaptive features: {use_adaptive}")
    
    rng = np.random.default_rng()
    
    global_best_budgets = None
    global_best_cost = float('inf')
    global_best_qubits = None
    global_best_runtime = None
    
    # Compute baseline for comparison
    baseline = np.array([1/3, 1/3, 1/3])
    baseline_dist = {
        'logical': baseline[0] * total_budget,
        't_states': baseline[1] * total_budget,
        'rotations': baseline[2] * total_budget
    }
    baseline_result = estimate_resources(logical_counts, baseline_dist)
    baseline_cost = compute_cost(
        baseline_result['physical_qubits'],
        baseline_result['runtime'],
        weight_qubits
    )
    
    print(f"\nBaseline (uniform):")
    print(f"  Qubits: {baseline_result['physical_qubits']:,}")
    print(f"  Runtime: {baseline_result['runtime']:.3e}")
    print(f"  Cost: {baseline_cost:.3e}")
    
    if use_adaptive:
        initial_points = generate_lhs_samples(n_restarts, 3, rng)
        print(f"\nUsing Latin Hypercube Sampling for diverse restarts")
    else:
        initial_points = None
    
    # Multiple restarts
    for restart in range(n_restarts):
        print(f"\n{'─'*70}")
        print(f"Restart {restart + 1}/{n_restarts}")
        
        if use_adaptive and initial_points is not None:
            current_budgets = initial_points[restart]
        else:
            current_budgets = rng.dirichlet(np.ones(3))
        
        current_dist = {
            'logical': current_budgets[0] * total_budget,
            't_states': current_budgets[1] * total_budget,
            'rotations': current_budgets[2] * total_budget
        }
        current_result = estimate_resources(logical_counts, current_dist)
        current_cost = compute_cost(
            current_result['physical_qubits'],
            current_result['runtime'],
            weight_qubits
        )
        
        best_budgets = current_budgets.copy()
        best_cost = current_cost
        best_qubits = current_result['physical_qubits']
        best_runtime = current_result['runtime']
        
        print(f"Start: [{current_budgets[0]:.3f}, {current_budgets[1]:.3f}, {current_budgets[2]:.3f}]")
        print(f"       Q={current_result['physical_qubits']:,}, R={current_result['runtime']:.3e}")
        
        temperature = initial_temp
        accepts = 0
        total_trials = 0
        cost_history = [current_cost]
        last_improvement = 0
        
        for iteration in range(max_iterations):
            if use_adaptive:
                neighbor_budgets = generate_neighbor_adaptive(
                    current_budgets, temperature, initial_temp, 
                    iteration, max_iterations, rng
                )
            else:
                temp_ratio = temperature / initial_temp
                step_size = 0.6 * temp_ratio + 0.2
                neighbor_budgets = current_budgets.copy()
                for i in range(3):
                    perturbation = rng.uniform(-step_size, step_size)
                    neighbor_budgets[i] += perturbation
                neighbor_budgets = np.maximum(neighbor_budgets, 0.01)
                neighbor_budgets = neighbor_budgets / np.sum(neighbor_budgets)
            
            neighbor_dist = {
                'logical': neighbor_budgets[0] * total_budget,
                't_states': neighbor_budgets[1] * total_budget,
                'rotations': neighbor_budgets[2] * total_budget
            }
            neighbor_result = estimate_resources(logical_counts, neighbor_dist)
            neighbor_cost = compute_cost(
                neighbor_result['physical_qubits'],
                neighbor_result['runtime'],
                weight_qubits
            )
            
            delta = neighbor_cost - current_cost
            
            if delta < 0:
                accept = True
                improvement = abs(delta / current_cost)
                last_improvement = improvement
            else:
                prob = np.exp(-delta / (temperature * current_cost))
                accept = rng.random() < prob
            
            total_trials += 1
            
            if accept:
                accepts += 1
                current_budgets = neighbor_budgets
                current_cost = neighbor_cost
                cost_history.append(current_cost)
                
                if current_cost < best_cost:
                    best_budgets = current_budgets.copy()
                    best_cost = current_cost
                    best_qubits = neighbor_result['physical_qubits']
                    best_runtime = neighbor_result['runtime']
            
            if use_adaptive:
                temperature = adaptive_cooling(
                    temperature, initial_temp, last_improvement, iteration
                )
            else:
                temperature *= 0.98
            
            if (iteration + 1) % 400 == 0:
                acc_rate = accepts / total_trials if total_trials > 0 else 0
                print(f"  Iter {iteration+1}: T={temperature:.4f}, "
                      f"Cost={best_cost:.3e}, AccRate={acc_rate:.2%}")
                accepts = 0
                total_trials = 0
            
            if use_adaptive and iteration > 200:
                if check_convergence(cost_history, window_size=50, threshold=0.0005):
                    print(f"  Converged at iteration {iteration+1}")
                    break
        
        if best_cost < global_best_cost:
            global_best_budgets = best_budgets.copy()
            global_best_cost = best_cost
            global_best_qubits = best_qubits
            global_best_runtime = best_runtime
            print(f"\n  ★ New best! Cost={global_best_cost:.3e}")
            print(f"    Q={global_best_qubits:,}, R={global_best_runtime:.3e}")
    
    print(f"\n{'='*70}")
    print("Optimization Complete")
    print(f"{'='*70}")
    print(f"Best cost: {global_best_cost:.3e}")
    print(f"Baseline cost: {baseline_cost:.3e}")
    improvement = ((baseline_cost - global_best_cost) / baseline_cost) * 100
    print(f"Cost improvement: {improvement:+.2f}%")
    print(f"\nOptimal budgets:")
    print(f"  Logical:   {global_best_budgets[0]:.4f} ({global_best_budgets[0]*total_budget:.6f})")
    print(f"  T-states:  {global_best_budgets[1]:.4f} ({global_best_budgets[1]*total_budget:.6f})")
    print(f"  Rotations: {global_best_budgets[2]:.4f} ({global_best_budgets[2]*total_budget:.6f})")
    
    return {
        'name': 'Enhanced_SA',
        'logical': global_best_budgets[0] * total_budget,
        't_states': global_best_budgets[1] * total_budget,
        'rotations': global_best_budgets[2] * total_budget
    }

def process_single_circuit(benchmark_name, circuit_size, total_error_budget=0.1, 
                          max_iterations=2000, n_restarts=10, verbose=False):
    """Process a single circuit and return results."""
    try:
        # Load benchmark
        qc = get_benchmark(
            benchmark=benchmark_name,
            level=BenchmarkLevel.ALG,
            circuit_size=circuit_size
        )
        
        # Transpile
        transpiled_qc = transpile(qc, basis_gates=QISKIT_STD_GATES, optimization_level=3)
        
        # Estimate logical counts
        estimation_result = estimate(transpiled_qc)
        logical_counts_dict = estimation_result["logicalCounts"]
        logical_counts = LogicalCounts(logical_counts_dict)
        
        # Run optimization (suppress output)
        import sys
        from io import StringIO
        
        if not verbose:
            old_stdout = sys.stdout
            sys.stdout = StringIO()
        
        optimal_distribution = simulated_annealing_optimization(
            logical_counts,
            total_error_budget,
            max_iterations=max_iterations,
            initial_temp=10.0,
            n_restarts=n_restarts,
            weight_qubits=0.5,
            use_adaptive=True
        )
        
        if not verbose:
            sys.stdout = old_stdout
        
        # Evaluate optimal
        optimal_result = estimate_resources(logical_counts, optimal_distribution)
        
        # Baseline - uniform distribution
        uniform_value = total_error_budget / 3
        uniform_distribution = {
            'logical': uniform_value,
            't_states': uniform_value,
            'rotations': uniform_value
        }
        uniform_result = estimate_resources(logical_counts, uniform_distribution)
        
        # Calculate improvements
        qubit_imp = ((uniform_result['physical_qubits'] - optimal_result['physical_qubits'])
                     / uniform_result['physical_qubits']) * 100
        runtime_imp = ((uniform_result['runtime'] - optimal_result['runtime'])
                       / uniform_result['runtime']) * 100
        metric_imp = ((uniform_result['metric'] - optimal_result['metric'])
                      / uniform_result['metric']) * 100
        
        return {
            'benchmark': benchmark_name,
            'circuit_size': circuit_size,
            'original_depth': qc.depth(),
            'transpiled_depth': transpiled_qc.depth(),
            'logical_qubits': logical_counts_dict['numQubits'],
            't_gates': logical_counts_dict['tCount'],
            'rotations': logical_counts_dict['rotationCount'],
            'rotation_depth': logical_counts_dict['rotationDepth'],
            'ccz_gates': logical_counts_dict['cczCount'],
            # Uniform distribution results
            'uniform_physical_qubits': uniform_result['physical_qubits'],
            'uniform_runtime': uniform_result['runtime'],
            'uniform_metric': uniform_result['metric'],
            # Optimal distribution budgets
            'optimal_logical_budget': optimal_distribution['logical'],
            'optimal_t_states_budget': optimal_distribution['t_states'],
            'optimal_rotations_budget': optimal_distribution['rotations'],
            # Optimal distribution results
            'optimal_physical_qubits': optimal_result['physical_qubits'],
            'optimal_runtime': optimal_result['runtime'],
            'optimal_metric': optimal_result['metric'],
            # Improvements
            'qubit_improvement_pct': qubit_imp,
            'runtime_improvement_pct': runtime_imp,
            'metric_improvement_pct': metric_imp,
            'status': 'success'
        }
        
    except Exception as e:
        return {
            'benchmark': benchmark_name,
            'circuit_size': circuit_size,
            'status': 'failed',
            'error': str(e)
        }

def main():
    import pandas as pd
    from datetime import datetime
    
    # ========================================================================
    # Define all benchmarks and circuit sizes to test
    # ========================================================================
    # MQT Bench contains many more benchmarks! Here's a comprehensive list:
    
    benchmarks_config = [
        # Quantum State Preparation
        ("ghz", [3, 4, 5, 6, 7, 8, 9, 10, 15, 16, 17, 18, 19, 20,21,22,23,24, 25, 30]),
        ("graphstate", [3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40]),
        ("wstate", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30]),
        
        # Quantum Algorithms
        ("qft", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,25,30]),
        ("qftentangled", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]),
        ("qpeexact", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,20,25]),
        ("qpeinexact", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,20,25]),
        ("dj", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30]),
        ("bv", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20,21,22,23,24, 25, 30]),
        ("grover", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
        
        
        # Variational Algorithms
        ("qaoa", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
        
        
        # Amplitude Estimation
        ("ae", [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
        
        # Quantum Chemistry
        ("hhl", [5, 7, 9, 11, 13, 15,17]),
        
        
        
        # Machine Learning
        ("qnn", [4, 5, 6, 7, 8, 9, 10,11,12,13,14,15]),

        #New Addition
        ("cdkm_ripple_carry_adder", [ 4, 8,12,14,16,18,20,22,24,26]),
        ("full_adder",[4,8,12,14,16,18,20,22,24,26,28]),
        ("half_adder",[3,5,7,9,11,13,15,17,19,21,23,25,27]),
        ("hrs_cumulative_multiplier",[5,9,13,17,21,25]),
        ("randomcircuit",[2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        ("vqe_su2",[2,3,4,5,6,7,8,9,10,15,20,25,30,35,40,80]),
        ("vqe_two_local",[2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        ("qwalk",[4,5,6,7,8,9,10,11,12,13,14,15]), 
        
        ("shor", [18,42]),
        ("modular_adder",[2,4,6,8,10,12,14,16,18,20,22,24,26,28,30]),
        ("vqe_real_amp",[2,3,4,5,6,7,8,9,10,11,12,13,14,15,35,40,45,50,55,60]),
        ("bmw_quark_cardinality",[2,3,4,5,10,11,12,13,14,15,40,60]),
        ("bmw_quark_copula",[2,4,6,8,10,12,14,16,18,20,28,30,50,60]),
        ("rg_qft_multiplier",[4,8,12,16,24,28,32]),
        ("draper_qft_adder",[2,4,6,8,10,14,16,18,20,22,24,26,28,30,40,50,60,90]),
        ("multiplier",[4,8,12,16,20,24,28,32,36,40,44,48,52,60]), #400
        ("vbe_ripple_carry_adder",[4,7,10,13,16,19,22,25,31,34,37,40,43,46,61,91]),
        
        
        
        
    ]
    
   
    
    print("="*70)
    print("MQT Benchmark Suite - Resource Estimation Analysis")
    print("="*70)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nTotal benchmarks: {len(benchmarks_config)}")
    total_circuits = sum(len(sizes) for _, sizes in benchmarks_config)
    print(f"Total circuits to process: {total_circuits}")
    print("="*70)
    
    # Collect all results
    all_results = []
    circuit_count = 0
    
    for benchmark_name, circuit_sizes in benchmarks_config:
        for circuit_size in circuit_sizes:
            circuit_count += 1
            print(f"\nProcessing [{circuit_count}/{total_circuits}]: {benchmark_name} (size={circuit_size})...", end=" ")
            
            result = process_single_circuit(
                benchmark_name=benchmark_name,
                circuit_size=circuit_size,
                total_error_budget=0.1,
                max_iterations=2000,
                n_restarts=10,
                verbose=False
            )
            
            all_results.append(result)
            
            if result['status'] == 'success':
                print(f"✓ (Metric improvement: {result['metric_improvement_pct']:+.2f}%)")
            else:
                print(f"✗ (Error: {result.get('error', 'Unknown')})")
    
    # ========================================================================
    # Save results to CSV
    # ========================================================================
    df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_filename = f'mqt_benchmark_results_{timestamp}.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"\n{'='*70}")
    print(f"Results saved to: {csv_filename}")
    print(f"{'='*70}")
    
    # ========================================================================
    # Calculate and display overall statistics
    # ========================================================================
    successful_results = df[df['status'] == 'success']
    failed_count = len(df) - len(successful_results)
    
    if len(successful_results) > 0:
        avg_qubit_imp = successful_results['qubit_improvement_pct'].mean()
        avg_runtime_imp = successful_results['runtime_improvement_pct'].mean()
        avg_metric_imp = successful_results['metric_improvement_pct'].mean()
        
        median_qubit_imp = successful_results['qubit_improvement_pct'].median()
        median_runtime_imp = successful_results['runtime_improvement_pct'].median()
        median_metric_imp = successful_results['metric_improvement_pct'].median()
        
        print(f"\n{'='*70}")
        print("OVERALL RESULTS - ENHANCED SIMULATED ANNEALING vs UNIFORM DISTRIBUTION")
        print(f"{'='*70}")
        print(f"\nTotal circuits processed: {len(df)}")
        print(f"  ✓ Successful: {len(successful_results)}")
        print(f"  ✗ Failed: {failed_count}")
        
        print(f"\n{'-'*70}")
        print("AVERAGE IMPROVEMENTS ACROSS ALL CIRCUITS:")
        print(f"{'-'*70}")
        print(f"Physical Qubits:  {avg_qubit_imp:+.2f}%")
        print(f"Runtime:          {avg_runtime_imp:+.2f}%")
        print(f"Overall Metric:   {avg_metric_imp:+.2f}%")
        
        print(f"\n{'-'*70}")
        print("MEDIAN IMPROVEMENTS ACROSS ALL CIRCUITS:")
        print(f"{'-'*70}")
        print(f"Physical Qubits:  {median_qubit_imp:+.2f}%")
        print(f"Runtime:          {median_runtime_imp:+.2f}%")
        print(f"Overall Metric:   {median_metric_imp:+.2f}%")
        
        print(f"\n{'-'*70}")
        print("IMPROVEMENT RANGE:")
        print(f"{'-'*70}")
        print(f"Physical Qubits:  [{successful_results['qubit_improvement_pct'].min():+.2f}%, {successful_results['qubit_improvement_pct'].max():+.2f}%]")
        print(f"Runtime:          [{successful_results['runtime_improvement_pct'].min():+.2f}%, {successful_results['runtime_improvement_pct'].max():+.2f}%]")
        print(f"Overall Metric:   [{successful_results['metric_improvement_pct'].min():+.2f}%, {successful_results['metric_improvement_pct'].max():+.2f}%]")
        
        # Best and worst performers
        best_metric = successful_results.loc[successful_results['metric_improvement_pct'].idxmax()]
        worst_metric = successful_results.loc[successful_results['metric_improvement_pct'].idxmin()]
        
        print(f"\n{'-'*70}")
        print("BEST PERFORMING CIRCUIT:")
        print(f"{'-'*70}")
        print(f"{best_metric['benchmark']} (size={int(best_metric['circuit_size'])})")
        print(f"  Metric improvement: {best_metric['metric_improvement_pct']:+.2f}%")
        print(f"  Qubit improvement:  {best_metric['qubit_improvement_pct']:+.2f}%")
        print(f"  Runtime improvement: {best_metric['runtime_improvement_pct']:+.2f}%")
        
        print(f"\n{'-'*70}")
        print("WORST PERFORMING CIRCUIT:")
        print(f"{'-'*70}")
        print(f"{worst_metric['benchmark']} (size={int(worst_metric['circuit_size'])})")
        print(f"  Metric improvement: {worst_metric['metric_improvement_pct']:+.2f}%")
        print(f"  Qubit improvement:  {worst_metric['qubit_improvement_pct']:+.2f}%")
        print(f"  Runtime improvement: {worst_metric['runtime_improvement_pct']:+.2f}%")
        
        print(f"\n{'='*70}")
        print(f"CONCLUSION:")
        print(f"{'='*70}")
        if avg_metric_imp > 0:
            print(f"✓ The Enhanced Simulated Annealing approach achieves an average")
            print(f"  improvement of {avg_metric_imp:.2f}% over uniform error budget")
            print(f"  distribution across {len(successful_results)} quantum circuits.")
        elif avg_metric_imp < 0:
            print(f"✗ The Enhanced Simulated Annealing approach shows an average")
            print(f"  degradation of {abs(avg_metric_imp):.2f}% compared to uniform")
            print(f"  error budget distribution across {len(successful_results)} circuits.")
        else:
            print(f"≈ The Enhanced Simulated Annealing approach performs similarly")
            print(f"  to uniform error budget distribution.")
        
    else:
        print("\n⚠ No successful results to analyze!")
    
    print(f"\n{'='*70}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

MQT Benchmark Suite - Resource Estimation Analysis
Start time: 2026-01-26 02:21:22

Total benchmarks: 31
Total circuits to process: 433

Processing [1/433]: ghz (size=3)... ✓ (Metric improvement: +0.00%)

Processing [2/433]: ghz (size=4)... ✓ (Metric improvement: +0.00%)

Processing [3/433]: ghz (size=5)... ✓ (Metric improvement: +0.00%)

Processing [4/433]: ghz (size=6)... ✓ (Metric improvement: +78.40%)

Processing [5/433]: ghz (size=7)... ✓ (Metric improvement: +78.40%)

Processing [6/433]: ghz (size=8)... ✓ (Metric improvement: +78.40%)

Processing [7/433]: ghz (size=9)... ✓ (Metric improvement: +78.40%)

Processing [8/433]: ghz (size=10)... ✓ (Metric improvement: +78.40%)

Processing [9/433]: ghz (size=15)... ✓ (Metric improvement: +0.00%)

Processing [10/433]: ghz (size=16)... ✓ (Metric improvement: +0.00%)

Processing [11/433]: ghz (size=17)... ✓ (Metric improvement: +0.00%)

Processing [12/433]: ghz (size=18)... ✓ (Metric improvement: +0.00%)

Processing [13/433]: ghz (size=19)